# 🎯 Python: Pandas Data Analytics

**Advanced Python Concepts for Interview Preparation**

- **Created:** 2026-02-28
- **Focus:** Vectorization, GroupBy, and High-Performance Data Manipulation
- **Tags:** `python` | `pandas` | `data-analysis` | `citi-prep`

## 📖 The Core Concept in Plain Language

Pandas is the industry standard for tabular data manipulation in Python. It provides the `DataFrame` object, which is conceptually similar to a SQL table or an Excel spreadsheet. But unlike writing manual Python loops ($O(N)$ overhead), Pandas delegates execution down to highly-optimized C code arrays (via NumPy).

### Why This Matters

- **Vectorization:** Doing math on entire columns instantly rather than iterating `for row in data:`.
- **Rich API:** Replacing 50 lines of Python loops with one `.groupby()` or `.merge()`.
- **Standardization:** If you know Pandas, you inherently understand the syntax for Spark DataFrames.

### The Key Insight

Never loop over a DataFrame (`iterrows` is strictly forbidden in production). Always think in vectors.

## 🔍 DataFrame Basics & Vectorization

In [ ]:
import pandas as pd
import numpy as np

# Create a DataFrame from a dictionary
data = {
    'date': ['2023-01-01', '2023-01-01', '2023-01-02', '2023-01-02'],
    'ticker': ['AAPL', 'MSFT', 'AAPL', 'MSFT'],
    'price': [150.0, 250.0, 155.0, 248.0],
    'volume': [1000, 2000, 1500, 1800]
}
df = pd.DataFrame(data)

# Vectorized Operation: Calculate total traded value instantly
# Notice there is no loop!
df['traded_value'] = df['price'] * df['volume']
print(df)

## 🏗️ The Anatomy of GroupBy

The split-apply-combine pattern is the most critical to master. You split the data into groups by some key, apply an aggregation function, and combine the results back into a DataFrame.

### Structure:
```python
df.groupby('category_column')['target_column'].agg(['sum', 'mean'])
```

In [ ]:
# Group by Ticker to find the average price and total volume
agg_df = df.groupby('ticker').agg({
    'price': 'mean',
    'volume': 'sum'
}).reset_index()

print(agg_df)

## 🔄 Apply vs Vectorization

`apply()` lets you run custom python functions on rows. But it is slow because it breaks out of C and back into Python for every single row. **Vectorization is always preferred.**

In [ ]:
# Goal: Create a new column 'status' if price > 200

# Approach 1: .apply() (Slow, effectively a loop under the hood)
def eval_price(p):
    return 'High' if p > 200 else 'Norm'
df['status_apply'] = df['price'].apply(eval_price)

# Approach 2: np.where (Vectorized, 100x faster)
df['status_vectorized'] = np.where(df['price'] > 200, 'High', 'Norm')

print("Both achieve the same, but np.where does it natively in C arrays.")

## 🌍 Real-World Example: Time-Series Merging

**The Citi Context:** We had files containing application downtime events, and files containing application deployment events. We needed to join them by Application ID, but also forward-fill the last known deployment version to see what version was active during the downtime.

### Pandas `merge_asof`
Standard `pd.merge()` does exact matches (like SQL JOIN). `pd.merge_asof()` performs a "fuzzy" join on ordered data (like timestamps).

In [ ]:
downtime = pd.DataFrame({'time': [10, 20, 30], 'app': ['A', 'A', 'B']})
deploys = pd.DataFrame({'time': [5, 25], 'app': ['A', 'A'], 'version': ['v1', 'v2']})

# We want to know what version was active during the downtime at time=20?
# Version v1 deployed at 5. Version v2 deployed at 25. So at 20, it was v1.

result = pd.merge_asof(
    downtime.sort_values('time'), 
    deploys.sort_values('time'), 
    on='time', 
    by='app',
    direction='backward' # use the closest deploy BEFORE the downtime
)
print(result)

## 🎭 Handling Missing Data (NaNs)

In [ ]:
df_nan = pd.DataFrame({'A': [1, np.nan, 3], 'B': ['foo', 'bar', np.nan]})
print("Original with NaNs:\n", df_nan)

# 1. Drop rows with any NaN
print("\nDropped:\n", df_nan.dropna())

# 2. Fill with specific values per column
print("\nFilled:\n", df_nan.fillna({'A': 0, 'B': 'unknown'}))

## 🎤 5 Interview Q&A

### Q1: What is the difference between a Series and a DataFrame?
**Answer:** A `Series` is a one-dimensional array with an index (representing a single column). A `DataFrame` is a two-dimensional tabular structure composed of a collection of Series that share the same index.

---

### Q2: Why is iterating over a DataFrame with `iterrows()` considered an anti-pattern?
**Answer:** Because `iterrows()` boxes every row into a new Python Series object, breaking out of the optimized C boundaries of the underlying NumPy arrays. It is notoriously slow. You should use vectorized numpy operations, `np.where`, or at worst, `itertuples()` if you strictly must iterate.

---

### Q3: How do you handle joining tables with different column names?
**Answer:** Use `pd.merge(df1, df2, left_on='id_tableA', right_on='uid_tableB', how='inner')`. 

---

### Q4: Contrast `iloc` and `loc`.
**Answer:** `loc[]` selects data by the *labels* of the index/columns. `iloc[]` selects data by the *integer position* (0 to N-1), exactly like standard list slicing.

---

### Q5: What happens to integer columns if an `np.nan` is introduced into them?
**Answer:** Traditionally, Pandas forces the entire integer column to upcast to `float64` because `NaN` is technically a floating-point type in NumPy. However, in modern Pandas, you can use the nullable integer type (`Int64` with a capital I) to avoid this.

## 📚 Key Terminology to Master

| Term | Definition | Example |
|------|-----------|----------|
| **Vectorization** | Performing operations on entire arrays at once via C | `df['A'] + df['B']` |
| **Index** | The row labels of the DataFrame. Crucial for alignment. | `df.set_index('date')` |
| **Broadcast** | Operations automatically expanding a scalar to fit an array | `df['A'] * 100` |
| **Split-Apply-Combine** | The underlying philosophy of `.groupby()` | `df.groupby('k').sum()` |
| **Method Chaining** | Stringing operations together conceptually | `df.dropna().groupby().mean()` |

## 💼 The Citi Narrative: Replacing 1000 Lines of Excel VBA

### The Problem
At Citi, reconciling two internal general ledgers involved an Excel macro that took 45 minutes to run across 300,000 rows. It looped line-by-line using `VLOOKUP` logic and `If...Then` macros.

### The Solution
I rewrote the extraction logic in pure Pandas using native vectorized operations. 
- Load CSVs -> `pd.read_csv()`
- VLOOKUP equivalent -> `pd.merge(how='outer', indicator=True)`
- Flag discrepancies -> `np.where(df['Amount_x'] != df['Amount_y'], 'Break', 'Match')`

### The Impact
The 45-minute VBA macro was replaced by a 14-line Python script that executed the exact same logic in 2.5 seconds. Reclaiming human execution time is the primary value of mastering Pandas.

## 💪 Practice Exercises

In [ ]:
# EXERCISE 1: Filtering
# How do you return only rows where 'ticker' is 'AAPL' and 'price' > 100?

print("df[(df['ticker'] == 'AAPL') & (df['price'] > 100)]")
print("NOTE: The parentheses are MANDATORY in Pandas bitwise boolean logic.")

In [ ]:
# EXERCISE 2: Aggregation differences
# What is the difference between df['A'].nunique() and df['A'].unique()?

print("nunique() returns the integer COUNT of distinct values. unique() returns the actual NumPy array of the distinct values.")

In [ ]:
# EXERCISE 3: Merge vs Concat
# When do you use pd.concat() instead of pd.merge()?

print("merge() is like SQL JOIN (horizontal bonding based on keys). concat() is like SQL UNION ALL (vertical stacking of rows, or appending columns blindly by index).")

## 🎯 Summary: Key Takeaways

### What You've Learned
1. ✅ **Vectorization over Loops:** `np.where()` is far preferred over `.apply()`, which is preferred over `.iterrows()`.
2. ✅ **GroupBy:** Think in terms of Split -> Apply -> Combine.
3. ✅ **Merge:** Standard SQL-like joins using `on`, `left_on`, `how='left'`.
4. ✅ **Time Series:** `merge_asof` allows you to align time-stamped tables that don't share exact seconds.

### Interview Confidence Checklist
- [ ] Can explain why DataFrames are faster than Python arrays.
- [ ] Can write a multi-column `.groupby().agg()` out loud.
- [ ] Understand the difference between `loc` and `iloc`.
- [ ] Understand how missing data (`NaN`) behaves in a generic SQL/Pandas context.
- [ ] Can tell the "VBA to Pandas 2-second" Citi story.

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

Pandas is the gateway. Spark is just Pandas mapped across 50 servers. 🚀